# 2.4 · DCF con incertidumbre

**Objetivo:** valorar una empresa con DCF y propagar la incertidumbre de los supuestos con Monte Carlo.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))  # repo root, so `common` is importable
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common import *
set_plot_style()

In [ ]:
fcf0 = 100.0        # FCF actual (M)
net_debt = 300.0    # deuda neta (M)
shares = 50.0       # acciones (M)
horizon = 5

## Ejercicio 1 — DCF determinista
Crecimiento 8%, WACC 9%, crecimiento terminal 2% (Gordon).

In [ ]:
def dcf_value_per_share(g, wacc, g_term):
    # TODO
    raise NotImplementedError


In [ ]:
def dcf_value_per_share(g, wacc, g_term):
    fcfs = np.array([fcf0 * (1 + g) ** t for t in range(1, horizon + 1)])
    disc = (1 + wacc) ** np.arange(1, horizon + 1)
    tv = fcfs[-1] * (1 + g_term) / (wacc - g_term) / (1 + wacc) ** horizon
    ev = (fcfs / disc).sum() + tv
    return (ev - net_debt) / shares

dcf_value_per_share(0.08, 0.09, 0.02)

## Ejercicio 2 — Monte Carlo
Distribuciones: $g\sim N(8\%,3\%)$, WACC $\sim N(9\%,1\%)$, $g_{term}\sim N(2\%,0.5\%)$. Descarta casos con WACC ≤ g_term. Reporta P5/P50/P95.

In [ ]:
rng = np.random.default_rng(1)
N = 20_000
g, wacc, gt = rng.normal(.08, .03, N), rng.normal(.09, .01, N), rng.normal(.02, .005, N)
ok = wacc > gt + 0.005
vals = np.array([dcf_value_per_share(a, b, c) for a, b, c in zip(g[ok], wacc[ok], gt[ok])])
p5, p50, p95 = np.percentile(vals, [5, 50, 95])
print(f"P5 {p5:.1f} | P50 {p50:.1f} | P95 {p95:.1f}")
plt.hist(vals, bins=80, alpha=.7); plt.axvline(p50, c="k"); plt.title("Valor por acción (DCF Monte Carlo)"); plt.show()